# BTK Datathon 2026 — Validation Strategy and Robust Modeling

This notebook is the third step after:

1. `01_data_understanding_baseline.ipynb`
2. `02_model_comparison.ipynb`

The previous notebook showed that `CatBoost_TFIDF` had the best local random CV score, but the Kaggle public score was worse than expected.

The goal here is **not only to reduce local CV MSE**, but to build a more reliable validation setup and generate more robust submissions.

Main ideas:

- Compare random KFold with recent-year validation.
- Detect whether models overfit the random CV split.
- Generate separate submissions for each model.
- Generate conservative blends, especially Ridge-weighted blends.
- Inspect prediction distributions before submitting.


## Why this notebook exists

Observed result:

- Local CV with `CatBoost_TFIDF`: around **79.76 MSE**
- Kaggle public score: around **90.24 MSE**

This gap suggests that random KFold may be too optimistic.  
The test set appears to be more concentrated in recent `application_year` values, so we need validation that checks recent-year generalization.


In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

RANDOM_STATE = 42
TARGET_COL = "career_success_score"
ID_COL = "student_id"
TEXT_COL = "mentor_feedback_text"

# Use fast mode first. After the notebook works, set RUN_FAST = False for stronger validation.
RUN_FAST = True
N_SPLITS = 3 if RUN_FAST else 5

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

print("N_SPLITS:", N_SPLITS)


N_SPLITS: 3


In [2]:
DATA_DIR = "../data"
SUBMISSION_DIR = "../submissions"
os.makedirs(SUBMISSION_DIR, exist_ok=True)

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test_x.csv")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

display(train.head())
display(test.head())


Train shape: (10000, 47)
Test shape: (10000, 46)
Sample submission shape: (2, 2)


,student_id,application_year,age,graduation_year,department,university_tier,cgpa,english_exam_score,attendance_rate,failed_courses_count,target_role,coding_score,problem_solving_score,data_structures_score,sql_score,machine_learning_score,backend_score,frontend_score,cloud_score,devops_score,project_quality_score,real_client_project_count,internship_count,internship_duration_months,freelance_project_count,hackathon_count,hackathon_awards,portfolio_score,github_repo_count,github_avg_stars,open_source_contribution_count,linkedin_profile_score,cv_quality_score,technical_interview_score,hr_interview_score,communication_score,teamwork_score,leadership_score,presentation_score,certification_count,bootcamp_count,applications_sent,interviews_attended,hobby,preferred_social_media_platform,career_success_score,mentor_feedback_text
0,STU_000001,2021,21,2021,Computer Engineering,Tier 4,3.17,62.54,77.31,0,DevOps Engineer,73.28,71.11,52.91,84.980000,81.77,62.710000,71.570000,63.041897,69.952625,81.90,0,3,11.0,0,0,0,65.54,18,1.85,10.0,86.58,42.06,40.57,50.29,79.83,44.14,62.70,58.84,3,1,24,0,photography,LinkedIn,86.78,Proje kalitesi ve makine öğrenimi konusundaki ...
1,STU_000002,2024,20,2024,Computer Engineering,Tier 4,3.24,75.10,87.13,3,Backend Developer,63.12,78.90,61.81,37.450740,65.54,69.944694,60.830000,64.510000,57.940000,24.68,0,0,NaN,1,1,0,54.48,7,1.22,1.0,33.34,65.39,82.99,67.43,43.60,22.05,42.32,40.54,2,0,46,5,reading,YouTube,46.16,Kodlama ve problem çözme becerileri gelişmekte...
2,STU_000003,2024,28,2024,Electrical Electronics Engineering,Tier 4,3.00,68.53,95.64,1,Frontend Developer,100.00,86.44,83.62,85.440000,87.18,80.580000,96.433149,62.220000,81.750000,78.92,2,0,0.0,2,0,0,75.10,4,12.12,2.0,61.37,52.25,43.06,20.19,48.62,65.64,47.27,82.56,1,2,46,5,cinema,Reddit,84.08,İleri düzey frontend geliştirme becerileri ile...
3,STU_000004,2019,22,2018,Computer Engineering,Tier 1,2.82,54.85,77.80,2,Backend Developer,99.08,72.15,77.15,89.214871,69.49,85.751415,72.860000,73.680000,54.080000,54.93,0,1,9.0,0,1,0,82.40,4,2.96,3.0,45.15,24.12,32.06,28.00,59.84,3.89,78.69,85.05,2,4,49,7,running,Reddit,89.97,Güçlü bir kodlama yeteneği ve backend geliştir...
4,STU_000005,2026,22,2026,Computer Engineering,Tier 3,2.28,72.25,71.97,1,Product Analyst,92.65,91.15,84.51,70.700000,74.11,80.620000,86.830000,80.340000,87.560000,72.85,2,0,NaN,2,0,0,48.02,14,0.97,12.0,74.86,74.83,71.82,65.14,63.30,52.86,27.22,84.29,1,0,119,13,football,X,92.46,Ürün analizi alanına olan tutkusu ve makine öğ...


,student_id,application_year,age,graduation_year,department,university_tier,cgpa,english_exam_score,attendance_rate,failed_courses_count,target_role,coding_score,problem_solving_score,data_structures_score,sql_score,machine_learning_score,backend_score,frontend_score,cloud_score,devops_score,project_quality_score,real_client_project_count,internship_count,internship_duration_months,freelance_project_count,hackathon_count,hackathon_awards,portfolio_score,github_repo_count,github_avg_stars,open_source_contribution_count,linkedin_profile_score,cv_quality_score,technical_interview_score,hr_interview_score,communication_score,teamwork_score,leadership_score,presentation_score,certification_count,bootcamp_count,applications_sent,interviews_attended,hobby,preferred_social_media_platform,mentor_feedback_text
0,STU_010001,2025,23,2025,Computer Engineering,Tier 4,2.59,45.27,79.11,2,Frontend Developer,42.60,61.53,61.17,34.780000,39.570000,59.61,39.143557,43.540000,52.800000,40.83,1,2,14.0,1,0,0,92.14,8,NaN,NaN,15.41,52.16,62.90,41.79,51.08,90.02,81.33,30.54,6,2,46,6,gaming,YouTube,Öğrencinin proje çalışmasında dikkat çekici bi...
1,STU_010002,2023,23,2022,Electrical Electronics Engineering,Tier 3,3.12,66.30,70.58,1,DevOps Engineer,71.63,95.42,81.59,78.360000,79.410000,74.65,60.360000,84.594122,66.284347,31.24,1,1,5.0,1,2,0,40.16,7,3.64,0.0,43.73,67.56,43.15,83.47,38.26,82.78,82.78,42.37,1,0,4,0,football,TikTok,"Bu öğrenci, problem çözme ve veri yapıları kon..."
2,STU_010003,2025,20,2025,Computer Engineering,Tier 3,3.48,75.45,67.93,0,Data Scientist,66.04,62.71,67.52,89.176084,58.108534,78.01,69.610000,66.840000,54.060000,26.53,1,1,2.0,3,0,0,59.12,8,NaN,NaN,14.96,68.73,71.74,64.96,88.49,51.79,63.30,66.01,4,1,66,3,cinema,Instagram,Öğrencinin veri bilimi konusundaki ilgisi ve g...
3,STU_010004,2020,26,2019,Software Engineering,Tier 2,2.92,45.58,83.95,0,Data Analyst,91.53,67.55,75.47,66.948792,62.822398,66.92,94.010000,64.990000,61.700000,82.96,1,0,0.0,0,1,0,73.90,9,3.18,3.0,NaN,52.79,84.60,79.40,76.32,90.89,46.64,82.60,3,2,42,2,chess,LinkedIn,Son derece etkileyici bir başarıya imza atan b...
4,STU_010005,2024,21,2024,Computer Engineering,Tier 3,2.85,63.08,84.18,1,Software Developer,70.82,70.65,65.16,62.290000,70.590000,54.44,80.420000,41.970000,63.230000,79.70,2,1,6.0,1,1,0,79.54,6,2.04,6.0,58.00,33.37,33.40,77.29,48.09,42.78,43.30,39.65,3,1,25,0,reading,TikTok,Öğrencinin yazılım geliştirme alanındaki çalış...


## 1. Check distribution shift

Here we compare train and test distributions, especially `application_year`.

If test is concentrated in recent years, random KFold can be misleading because it mixes old and recent years in every fold.


In [3]:
print("Train application_year distribution:")
display(train["application_year"].value_counts().sort_index().to_frame("train_count"))

print("Test application_year distribution:")
display(test["application_year"].value_counts().sort_index().to_frame("test_count"))

year_summary = train.groupby("application_year")[TARGET_COL].agg(["count", "mean", "std", "min", "max"])
display(year_summary)

dist = pd.concat([
    train["application_year"].value_counts(normalize=True).sort_index().rename("train_share"),
    test["application_year"].value_counts(normalize=True).sort_index().rename("test_share")
], axis=1).fillna(0)

dist["test_minus_train"] = dist["test_share"] - dist["train_share"]
display(dist)


Train application_year distribution:


,train_count
application_year,
2019,1289
2020,1329
2021,1270
2022,1293
2023,1312
2024,1319
2025,1191
2026,997


Test application_year distribution:


,test_count
application_year,
2019,403
2020,507
2021,691
2022,881
2023,1298
2024,1994
2025,2197
2026,2029


,count,mean,std,min,max
application_year,,,,,
2019,1289,77.942583,12.580981,33.14,100.0
2020,1329,77.780609,12.935411,24.93,100.0
2021,1270,77.392386,14.171235,27.46,100.0
2022,1293,77.569791,14.292887,31.04,100.0
2023,1312,76.785343,15.420235,26.62,100.0
2024,1319,77.661077,15.403322,18.04,100.0
2025,1191,75.472427,18.317177,15.23,100.0
2026,997,74.158064,18.023350,0.00,100.0


,train_share,test_share,test_minus_train
application_year,,,
2019,0.1289,0.0403,-0.0886
2020,0.1329,0.0507,-0.0822
2021,0.1270,0.0691,-0.0579
2022,0.1293,0.0881,-0.0412
2023,0.1312,0.1298,-0.0014
2024,0.1319,0.1994,0.0675
2025,0.1191,0.2197,0.1006
2026,0.0997,0.2029,0.1032


## 2. Safe feature engineering

Only row-wise transformations are used here.  
We avoid fitting text vectorizers, encoders, imputers, or scalers on train+test together.

That part must happen inside the sklearn pipeline so each CV fold is fitted only on its training split.


In [4]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    skill_cols = [
        "coding_score", "problem_solving_score", "data_structures_score", "sql_score",
        "machine_learning_score", "backend_score", "frontend_score", "cloud_score", "devops_score"
    ]
    existing_skill_cols = [c for c in skill_cols if c in df.columns]

    soft_cols = ["communication_score", "teamwork_score", "leadership_score", "presentation_score"]
    existing_soft_cols = [c for c in soft_cols if c in df.columns]

    interview_cols = ["technical_interview_score", "hr_interview_score"]
    existing_interview_cols = [c for c in interview_cols if c in df.columns]

    portfolio_cols = [
        "portfolio_score", "github_repo_count", "github_avg_stars",
        "open_source_contribution_count", "linkedin_profile_score", "cv_quality_score"
    ]
    existing_portfolio_cols = [c for c in portfolio_cols if c in df.columns]

    experience_cols = [
        "real_client_project_count", "internship_count", "freelance_project_count",
        "hackathon_count", "hackathon_awards", "bootcamp_count", "certification_count"
    ]
    existing_experience_cols = [c for c in experience_cols if c in df.columns]

    # Aggregate skill signals
    if existing_skill_cols:
        df["technical_skill_mean"] = df[existing_skill_cols].mean(axis=1)
        df["technical_skill_std"] = df[existing_skill_cols].std(axis=1)
        df["technical_skill_min"] = df[existing_skill_cols].min(axis=1)
        df["technical_skill_max"] = df[existing_skill_cols].max(axis=1)
        df["technical_skill_range"] = df["technical_skill_max"] - df["technical_skill_min"]

    if existing_soft_cols:
        df["soft_skill_mean"] = df[existing_soft_cols].mean(axis=1)
        df["soft_skill_std"] = df[existing_soft_cols].std(axis=1)

    if existing_interview_cols:
        df["interview_mean"] = df[existing_interview_cols].mean(axis=1)
        if set(["technical_interview_score", "hr_interview_score"]).issubset(df.columns):
            df["interview_gap_technical_minus_hr"] = df["technical_interview_score"] - df["hr_interview_score"]

    if existing_portfolio_cols:
        df["portfolio_readiness_mean"] = df[existing_portfolio_cols].mean(axis=1)

    if existing_experience_cols:
        df["experience_total"] = df[existing_experience_cols].sum(axis=1)

    if set(["internship_duration_months", "internship_count"]).issubset(df.columns):
        df["avg_internship_duration"] = df["internship_duration_months"] / (df["internship_count"] + 1)

    if set(["applications_sent", "interviews_attended"]).issubset(df.columns):
        df["interview_rate"] = df["interviews_attended"] / (df["applications_sent"] + 1)

    if set(["application_year", "graduation_year"]).issubset(df.columns):
        df["years_to_graduation"] = df["graduation_year"] - df["application_year"]

    if set(["application_year", "age"]).issubset(df.columns):
        df["birth_year_estimate"] = df["application_year"] - df["age"]

    # Role-aligned skill signal
    # This is intentionally simple and transparent.
    role = df["target_role"].astype(str).str.lower() if "target_role" in df.columns else pd.Series("", index=df.index)
    df["role_focus_score"] = np.nan

    if "machine_learning_score" in df.columns:
        mask = role.str.contains("data scientist|machine learning|ml|ai", regex=True, na=False)
        df.loc[mask, "role_focus_score"] = df.loc[mask, "machine_learning_score"]

    if "sql_score" in df.columns:
        mask = role.str.contains("data analyst|analytics|business intelligence|bi", regex=True, na=False)
        df.loc[mask, "role_focus_score"] = df.loc[mask, "sql_score"]

    if set(["backend_score", "devops_score", "cloud_score"]).issubset(df.columns):
        mask = role.str.contains("backend|devops|cloud", regex=True, na=False)
        df.loc[mask, "role_focus_score"] = df.loc[mask, ["backend_score", "devops_score", "cloud_score"]].mean(axis=1)

    if set(["frontend_score", "backend_score"]).issubset(df.columns):
        mask = role.str.contains("software|full stack|fullstack|developer|engineer", regex=True, na=False)
        df.loc[mask, "role_focus_score"] = df.loc[mask, ["frontend_score", "backend_score"]].mean(axis=1)

    if "technical_skill_mean" in df.columns:
        df["role_focus_score"] = df["role_focus_score"].fillna(df["technical_skill_mean"])

    # Text length features
    if TEXT_COL in df.columns:
        text = df[TEXT_COL].fillna("").astype(str)
        df["mentor_text_char_len"] = text.str.len()
        df["mentor_text_word_len"] = text.str.split().str.len()

    # Missingness can be signal in synthetic/competition datasets
    base_cols_for_missing = [c for c in df.columns if c not in [TARGET_COL]]
    df["missing_value_count"] = df[base_cols_for_missing].isna().sum(axis=1)

    return df


train_fe = add_features(train)
test_fe = add_features(test)

print("Train FE shape:", train_fe.shape)
print("Test FE shape:", test_fe.shape)

display(train_fe.head())


Train FE shape: (10000, 66)
Test FE shape: (10000, 65)


,student_id,application_year,age,graduation_year,department,university_tier,cgpa,english_exam_score,attendance_rate,failed_courses_count,target_role,coding_score,problem_solving_score,data_structures_score,sql_score,machine_learning_score,backend_score,frontend_score,cloud_score,devops_score,project_quality_score,real_client_project_count,internship_count,internship_duration_months,freelance_project_count,hackathon_count,hackathon_awards,portfolio_score,github_repo_count,github_avg_stars,open_source_contribution_count,linkedin_profile_score,cv_quality_score,technical_interview_score,hr_interview_score,communication_score,teamwork_score,leadership_score,presentation_score,certification_count,bootcamp_count,applications_sent,interviews_attended,hobby,preferred_social_media_platform,career_success_score,mentor_feedback_text,technical_skill_mean,technical_skill_std,technical_skill_min,technical_skill_max,technical_skill_range,soft_skill_mean,soft_skill_std,interview_mean,interview_gap_technical_minus_hr,portfolio_readiness_mean,experience_total,avg_internship_duration,interview_rate,years_to_graduation,birth_year_estimate,role_focus_score,mentor_text_char_len,mentor_text_word_len,missing_value_count
0,STU_000001,2021,21,2021,Computer Engineering,Tier 4,3.17,62.54,77.31,0,DevOps Engineer,73.28,71.11,52.91,84.980000,81.77,62.710000,71.570000,63.041897,69.952625,81.90,0,3,11.0,0,0,0,65.54,18,1.85,10.0,86.58,42.06,40.57,50.29,79.83,44.14,62.70,58.84,3,1,24,0,photography,LinkedIn,86.78,Proje kalitesi ve makine öğrenimi konusundaki ...,70.147169,9.815953,52.91000,84.98,32.07000,61.3775,14.672129,45.430,-9.72,37.338333,7,2.75,0.000000,0,2000,67.140000,225,30,0
1,STU_000002,2024,20,2024,Computer Engineering,Tier 4,3.24,75.10,87.13,3,Backend Developer,63.12,78.90,61.81,37.450740,65.54,69.944694,60.830000,64.510000,57.940000,24.68,0,0,NaN,1,1,0,54.48,7,1.22,1.0,33.34,65.39,82.99,67.43,43.60,22.05,42.32,40.54,2,0,46,5,reading,YouTube,46.16,Kodlama ve problem çözme becerileri gelişmekte...,62.227270,11.118139,37.45074,78.90,41.44926,37.1275,10.129684,75.210,15.56,27.071667,4,NaN,0.106383,0,2004,65.387347,259,33,2
2,STU_000003,2024,28,2024,Electrical Electronics Engineering,Tier 4,3.00,68.53,95.64,1,Frontend Developer,100.00,86.44,83.62,85.440000,87.18,80.580000,96.433149,62.220000,81.750000,78.92,2,0,0.0,2,0,0,75.10,4,12.12,2.0,61.37,52.25,43.06,20.19,48.62,65.64,47.27,82.56,1,2,46,5,cinema,Reddit,84.08,İleri düzey frontend geliştirme becerileri ile...,84.851461,10.685677,62.22000,100.00,37.78000,61.0225,16.614637,31.625,22.87,34.473333,7,0.00,0.106383,0,1996,88.506575,232,28,0
3,STU_000004,2019,22,2018,Computer Engineering,Tier 1,2.82,54.85,77.80,2,Backend Developer,99.08,72.15,77.15,89.214871,69.49,85.751415,72.860000,73.680000,54.080000,54.93,0,1,9.0,0,1,0,82.40,4,2.96,3.0,45.15,24.12,32.06,28.00,59.84,3.89,78.69,85.05,2,4,49,7,running,Reddit,89.97,Güçlü bir kodlama yeteneği ve backend geliştir...,77.050698,12.974625,54.08000,99.08,45.00000,56.8675,36.904950,30.030,4.06,26.938333,8,4.50,0.140000,-1,1997,79.305707,227,29,0
4,STU_000005,2026,22,2026,Computer Engineering,Tier 3,2.28,72.25,71.97,1,Product Analyst,92.65,91.15,84.51,70.700000,74.11,80.620000,86.830000,80.340000,87.560000,72.85,2,0,NaN,2,0,0,48.02,14,0.97,12.0,74.86,74.83,71.82,65.14,63.30,52.86,27.22,84.29,1,0,119,13,football,X,92.46,Ürün analizi alanına olan tutkusu ve makine öğ...,83.163333,7.417122,70.70000,92.65,21.95000,56.9175,23.723370,68.480,6.68,37.446667,5,NaN,0.108333,0,2004,83.163333,225,27,2


In [5]:
X = train_fe.drop(columns=[TARGET_COL])
y = train_fe[TARGET_COL].copy()
X_test = test_fe.copy()
test_ids = X_test[ID_COL].copy()

drop_cols = [ID_COL]
X = X.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

# Make sure train/test have the same feature columns
missing_in_test = sorted(set(X.columns) - set(X_test.columns))
missing_in_train = sorted(set(X_test.columns) - set(X.columns))

print("Missing in test:", missing_in_test)
print("Missing in train:", missing_in_train)

X_test = X_test[X.columns]

numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

# Keep text separate from normal categoricals
if TEXT_COL in categorical_cols:
    categorical_cols.remove(TEXT_COL)

print("Numeric cols:", len(numeric_cols))
print("Categorical cols:", len(categorical_cols))
print("Text col:", TEXT_COL)


Missing in test: []
Missing in train: []
Numeric cols: 58
Categorical cols: 5
Text col: mentor_feedback_text


## 3. Preprocessing and model definitions

Important detail for text:

`ColumnTransformer` must receive the text column as `[TEXT_COL]`, and the text pipeline converts the imputed 2D array back to 1D before `TfidfVectorizer`.


In [6]:
def make_preprocessor(tfidf_max_features=3000):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    text_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="")),
        ("to_1d", FunctionTransformer(lambda x: x.ravel(), validate=False)),
        ("tfidf", TfidfVectorizer(
            max_features=tfidf_max_features,
            ngram_range=(1, 2),
            min_df=2
        ))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
            ("text", text_transformer, [TEXT_COL])
        ],
        remainder="drop"
    )


def build_models():
    models = {}

    models["Ridge_TFIDF"] = Pipeline(steps=[
        ("preprocess", make_preprocessor(tfidf_max_features=3500)),
        ("model", Ridge(alpha=10.0))
    ])

    models["ElasticNet_TFIDF"] = Pipeline(steps=[
        ("preprocess", make_preprocessor(tfidf_max_features=3000)),
        ("model", ElasticNet(alpha=0.001, l1_ratio=0.05, max_iter=5000, random_state=RANDOM_STATE))
    ])

    try:
        from lightgbm import LGBMRegressor
        models["LightGBM_TFIDF"] = Pipeline(steps=[
            ("preprocess", make_preprocessor(tfidf_max_features=2500)),
            ("model", LGBMRegressor(
                n_estimators=600 if RUN_FAST else 1000,
                learning_rate=0.035,
                num_leaves=31,
                max_depth=-1,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1
            ))
        ])
    except Exception as e:
        print("LightGBM unavailable:", e)

    try:
        from xgboost import XGBRegressor
        models["XGBoost_TFIDF"] = Pipeline(steps=[
            ("preprocess", make_preprocessor(tfidf_max_features=2500)),
            ("model", XGBRegressor(
                n_estimators=500 if RUN_FAST else 900,
                learning_rate=0.035,
                max_depth=3,
                min_child_weight=3,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_alpha=0.1,
                reg_lambda=1.0,
                objective="reg:squarederror",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist"
            ))
        ])
    except Exception as e:
        print("XGBoost unavailable:", e)

    try:
        from catboost import CatBoostRegressor
        models["CatBoost_TFIDF"] = Pipeline(steps=[
            ("preprocess", make_preprocessor(tfidf_max_features=2500)),
            ("model", CatBoostRegressor(
                iterations=500 if RUN_FAST else 900,
                learning_rate=0.04,
                depth=4,
                l2_leaf_reg=8.0,
                random_strength=1.0,
                loss_function="RMSE",
                random_seed=RANDOM_STATE,
                verbose=False
            ))
        ])
    except Exception as e:
        print("CatBoost unavailable:", e)

    return models


models = build_models()
print("Models:", list(models.keys()))


Models: ['Ridge_TFIDF', 'ElasticNet_TFIDF', 'LightGBM_TFIDF', 'XGBoost_TFIDF', 'CatBoost_TFIDF']


## 4. Random KFold validation

This is useful, but it may be too optimistic if test distribution is shifted.


In [7]:
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

random_results = []

for name, model in models.items():
    print(f"\nRunning random CV for {name}...")

    scores = -cross_val_score(
        model,
        X,
        y,
        scoring="neg_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        error_score="raise"
    )

    random_results.append({
        "model": name,
        "random_cv_mse": scores.mean(),
        "random_cv_rmse": np.sqrt(scores.mean()),
        "random_cv_std": scores.std(),
        "random_fold_scores": scores
    })

    print("Fold MSE:", np.round(scores, 4))
    print(f"Mean MSE: {scores.mean():.4f} | RMSE: {np.sqrt(scores.mean()):.4f} | Std: {scores.std():.4f}")

random_results_df = pd.DataFrame(random_results).sort_values("random_cv_mse")
display(random_results_df[["model", "random_cv_mse", "random_cv_rmse", "random_cv_std"]])



Running random CV for Ridge_TFIDF...
Fold MSE: [91.5336 86.1265 87.7356]
Mean MSE: 88.4652 | RMSE: 9.4056 | Std: 2.2669

Running random CV for ElasticNet_TFIDF...
Fold MSE: [91.8502 86.0844 88.0744]
Mean MSE: 88.6696 | RMSE: 9.4165 | Std: 2.3912

Running random CV for LightGBM_TFIDF...
Fold MSE: [88.4901 80.6497 82.7519]
Mean MSE: 83.9639 | RMSE: 9.1632 | Std: 3.3136

Running random CV for XGBoost_TFIDF...
Fold MSE: [88.4496 83.0587 83.9848]
Mean MSE: 85.1644 | RMSE: 9.2285 | Std: 2.3536

Running random CV for CatBoost_TFIDF...
Fold MSE: [87.3064 81.608  84.0448]
Mean MSE: 84.3198 | RMSE: 9.1826 | Std: 2.3345


,model,random_cv_mse,random_cv_rmse,random_cv_std
2,LightGBM_TFIDF,83.963872,9.163180,3.313585
4,CatBoost_TFIDF,84.319753,9.182579,2.334473
3,XGBoost_TFIDF,85.164368,9.228454,2.353600
0,Ridge_TFIDF,88.465237,9.405596,2.266928
1,ElasticNet_TFIDF,88.669641,9.416456,2.391232


## 5. Recent-year validation

This is the important check after the Kaggle/public mismatch.

We train on earlier years and validate on 2025–2026.  
This is usually harsher than random KFold, but it may better represent the shifted test set.


In [8]:
recent_valid_mask = train_fe["application_year"] >= 2025
recent_train_mask = ~recent_valid_mask

X_recent_train = X.loc[recent_train_mask].copy()
y_recent_train = y.loc[recent_train_mask].copy()
X_recent_valid = X.loc[recent_valid_mask].copy()
y_recent_valid = y.loc[recent_valid_mask].copy()

print("Recent train shape:", X_recent_train.shape)
print("Recent valid shape:", X_recent_valid.shape)
print("Recent valid years:")
display(train_fe.loc[recent_valid_mask, "application_year"].value_counts().sort_index())


Recent train shape: (7812, 64)
Recent valid shape: (2188, 64)
Recent valid years:


application_year
2025    1191
2026     997
Name: count, dtype: int64

In [9]:
recent_results = []
recent_valid_predictions = {}

for name, model in models.items():
    print(f"\nRunning recent-year validation for {name}...")

    fitted = clone(model)
    fitted.fit(X_recent_train, y_recent_train)

    preds = fitted.predict(X_recent_valid)
    preds = np.clip(preds, 0, 100)

    mse = mean_squared_error(y_recent_valid, preds)

    recent_valid_predictions[name] = preds

    recent_results.append({
        "model": name,
        "recent_year_mse": mse,
        "recent_year_rmse": np.sqrt(mse),
        "pred_mean": preds.mean(),
        "pred_std": preds.std(),
        "pred_min": preds.min(),
        "pred_max": preds.max()
    })

    print(f"Recent-year MSE: {mse:.4f} | RMSE: {np.sqrt(mse):.4f}")
    print(f"Pred mean: {preds.mean():.4f} | std: {preds.std():.4f} | min: {preds.min():.4f} | max: {preds.max():.4f}")

recent_results_df = pd.DataFrame(recent_results).sort_values("recent_year_mse")
display(recent_results_df)



Running recent-year validation for Ridge_TFIDF...
Recent-year MSE: 137.7705 | RMSE: 11.7376
Pred mean: 76.1722 | std: 11.2554 | min: 38.1417 | max: 100.0000

Running recent-year validation for ElasticNet_TFIDF...
Recent-year MSE: 137.3970 | RMSE: 11.7216
Pred mean: 76.1514 | std: 11.3148 | min: 37.7687 | max: 100.0000

Running recent-year validation for LightGBM_TFIDF...
Recent-year MSE: 131.8728 | RMSE: 11.4836
Pred mean: 77.3840 | std: 11.8057 | min: 41.6924 | max: 100.0000

Running recent-year validation for XGBoost_TFIDF...
Recent-year MSE: 137.9448 | RMSE: 11.7450
Pred mean: 77.5678 | std: 10.9731 | min: 44.7669 | max: 100.0000

Running recent-year validation for CatBoost_TFIDF...
Recent-year MSE: 136.4271 | RMSE: 11.6802
Pred mean: 77.5440 | std: 10.9748 | min: 45.3433 | max: 100.0000


,model,recent_year_mse,recent_year_rmse,pred_mean,pred_std,pred_min,pred_max
2,LightGBM_TFIDF,131.872759,11.483586,77.384011,11.805707,41.692431,100.0
4,CatBoost_TFIDF,136.427060,11.680199,77.544021,10.974781,45.343276,100.0
1,ElasticNet_TFIDF,137.397042,11.721648,76.151432,11.314843,37.768747,100.0
0,Ridge_TFIDF,137.770493,11.737568,76.172170,11.255367,38.141703,100.0
3,XGBoost_TFIDF,137.944771,11.744989,77.567848,10.973091,44.766872,100.0


## 6. Compare random CV vs recent-year validation

Models that look very good in random CV but much worse in recent-year validation are less reliable for Kaggle.


In [10]:
comparison_df = random_results_df.merge(
    recent_results_df[["model", "recent_year_mse", "recent_year_rmse"]],
    on="model",
    how="left"
)

comparison_df["recent_minus_random"] = comparison_df["recent_year_mse"] - comparison_df["random_cv_mse"]
comparison_df = comparison_df.sort_values(["recent_year_mse", "random_cv_mse"])

display(comparison_df[[
    "model",
    "random_cv_mse",
    "recent_year_mse",
    "recent_minus_random",
    "random_cv_rmse",
    "recent_year_rmse"
]])


,model,random_cv_mse,recent_year_mse,recent_minus_random,random_cv_rmse,recent_year_rmse
0,LightGBM_TFIDF,83.963872,131.872759,47.908887,9.163180,11.483586
1,CatBoost_TFIDF,84.319753,136.427060,52.107307,9.182579,11.680199
4,ElasticNet_TFIDF,88.669641,137.397042,48.727402,9.416456,11.721648
3,Ridge_TFIDF,88.465237,137.770493,49.305256,9.405596,11.737568
2,XGBoost_TFIDF,85.164368,137.944771,52.780403,9.228454,11.744989


## 7. Fit each model on full train and save individual submissions

We do this because local-best model is not always Kaggle-best model.

Recommended submit order:

1. Ridge
2. LightGBM
3. XGBoost
4. CatBoost
5. Conservative blend


In [11]:
test_predictions = {}

for name, model in models.items():
    print(f"\nFitting full train for {name}...")

    fitted = clone(model)
    fitted.fit(X, y)

    preds = fitted.predict(X_test)
    preds = np.clip(preds, 0, 100)
    test_predictions[name] = preds

    submission = pd.DataFrame({
        ID_COL: test_ids,
        TARGET_COL: preds
    })

    path = os.path.join(SUBMISSION_DIR, f"submission_03_{name}.csv")
    submission.to_csv(path, index=False)
    print("Saved:", path)
    print(f"mean={preds.mean():.4f}, std={preds.std():.4f}, min={preds.min():.4f}, max={preds.max():.4f}")



Fitting full train for Ridge_TFIDF...
Saved: ../submissions\submission_03_Ridge_TFIDF.csv
mean=76.0435, std=11.7798, min=33.6086, max=100.0000

Fitting full train for ElasticNet_TFIDF...
Saved: ../submissions\submission_03_ElasticNet_TFIDF.csv
mean=76.0393, std=11.7899, min=33.6293, max=100.0000

Fitting full train for LightGBM_TFIDF...
Saved: ../submissions\submission_03_LightGBM_TFIDF.csv
mean=76.1433, std=12.6064, min=34.1938, max=100.0000

Fitting full train for XGBoost_TFIDF...
Saved: ../submissions\submission_03_XGBoost_TFIDF.csv
mean=76.2292, std=11.7459, min=35.8500, max=100.0000

Fitting full train for CatBoost_TFIDF...
Saved: ../submissions\submission_03_CatBoost_TFIDF.csv
mean=76.2131, std=11.8881, min=39.4309, max=100.0000


## 8. Blend candidates

Because CatBoost public score was worse than expected, we create conservative blends.

The most important one is `conservative_ridge_blend`, which gives more weight to Ridge because linear models often generalize better under distribution shift.


In [12]:
def weighted_blend(pred_dict, weights):
    available = {k: v for k, v in weights.items() if k in pred_dict}
    total_weight = sum(available.values())
    if total_weight == 0:
        raise ValueError("No available models for this blend.")

    blend = np.zeros(len(next(iter(pred_dict.values()))))
    for name, weight in available.items():
        blend += (weight / total_weight) * pred_dict[name]

    return np.clip(blend, 0, 100)


blend_recipes = {
    "conservative_ridge_blend": {
        "Ridge_TFIDF": 0.55,
        "LightGBM_TFIDF": 0.20,
        "XGBoost_TFIDF": 0.15,
        "CatBoost_TFIDF": 0.10
    },
    "balanced_blend": {
        "Ridge_TFIDF": 0.35,
        "LightGBM_TFIDF": 0.25,
        "XGBoost_TFIDF": 0.20,
        "CatBoost_TFIDF": 0.20
    },
    "tree_light_blend": {
        "Ridge_TFIDF": 0.40,
        "LightGBM_TFIDF": 0.30,
        "XGBoost_TFIDF": 0.20,
        "CatBoost_TFIDF": 0.10
    }
}

blend_predictions = {}

for blend_name, weights in blend_recipes.items():
    preds = weighted_blend(test_predictions, weights)
    blend_predictions[blend_name] = preds

    submission = pd.DataFrame({
        ID_COL: test_ids,
        TARGET_COL: preds
    })

    path = os.path.join(SUBMISSION_DIR, f"submission_03_{blend_name}.csv")
    submission.to_csv(path, index=False)
    print("Saved:", path)
    print(f"{blend_name}: mean={preds.mean():.4f}, std={preds.std():.4f}, min={preds.min():.4f}, max={preds.max():.4f}")


Saved: ../submissions\submission_03_conservative_ridge_blend.csv
conservative_ridge_blend: mean=76.1083, std=11.7641, min=35.1246, max=100.0000
Saved: ../submissions\submission_03_balanced_blend.csv
balanced_blend: mean=76.1395, std=11.8153, min=36.1258, max=100.0000
Saved: ../submissions\submission_03_tree_light_blend.csv
tree_light_blend: mean=76.1275, std=11.8409, min=35.4305, max=100.0000


## 9. Optional year-prior smoothing

This is optional and should not be blindly trusted.

It blends model predictions slightly with the historical average target of each `application_year`.

This can help if the public test set is heavily shifted by year, but it may also hurt if the model already captured the year effect.


In [13]:
year_mean = train.groupby("application_year")[TARGET_COL].mean()
global_mean = train[TARGET_COL].mean()

year_prior = test["application_year"].map(year_mean).fillna(global_mean).values

year_smoothed_predictions = {}

# Use conservative blend as the base for smoothing
if "conservative_ridge_blend" in blend_predictions:
    base_pred = blend_predictions["conservative_ridge_blend"]

    for alpha in [0.05, 0.10, 0.15]:
        # alpha = how much we trust the year prior
        preds = (1 - alpha) * base_pred + alpha * year_prior
        preds = np.clip(preds, 0, 100)

        name = f"conservative_blend_year_smooth_{int(alpha*100)}"
        year_smoothed_predictions[name] = preds

        submission = pd.DataFrame({
            ID_COL: test_ids,
            TARGET_COL: preds
        })

        path = os.path.join(SUBMISSION_DIR, f"submission_03_{name}.csv")
        submission.to_csv(path, index=False)
        print("Saved:", path)
        print(f"{name}: mean={preds.mean():.4f}, std={preds.std():.4f}, min={preds.min():.4f}, max={preds.max():.4f}")


Saved: ../submissions\submission_03_conservative_blend_year_smooth_5.csv
conservative_blend_year_smooth_5: mean=76.1202, std=11.1810, min=37.0763, max=98.8971
Saved: ../submissions\submission_03_conservative_blend_year_smooth_10.csv
conservative_blend_year_smooth_10: mean=76.1321, std=10.5983, min=39.0280, max=97.7943
Saved: ../submissions\submission_03_conservative_blend_year_smooth_15.csv
conservative_blend_year_smooth_15: mean=76.1440, std=10.0161, min=40.9796, max=96.6914


## 10. Prediction distribution diagnostics

Before submitting, check whether predictions are too compressed or too extreme.

MSE punishes large errors heavily.  
If predictions are too compressed, the model may miss low/high true scores.  
If predictions are too extreme, public score may explode.


In [14]:
all_prediction_sets = {}
all_prediction_sets.update(test_predictions)
all_prediction_sets.update(blend_predictions)
all_prediction_sets.update(year_smoothed_predictions)

diagnostics = []

for name, preds in all_prediction_sets.items():
    diagnostics.append({
        "name": name,
        "mean": np.mean(preds),
        "std": np.std(preds),
        "min": np.min(preds),
        "p01": np.percentile(preds, 1),
        "p05": np.percentile(preds, 5),
        "p25": np.percentile(preds, 25),
        "p50": np.percentile(preds, 50),
        "p75": np.percentile(preds, 75),
        "p95": np.percentile(preds, 95),
        "p99": np.percentile(preds, 99),
        "max": np.max(preds),
        "n_equal_100": int(np.sum(preds >= 100)),
        "n_below_40": int(np.sum(preds < 40)),
        "n_below_50": int(np.sum(preds < 50))
    })

diagnostics_df = pd.DataFrame(diagnostics).sort_values("std", ascending=False)

print("Train target distribution:")
display(train[TARGET_COL].describe())

print("Prediction diagnostics:")
display(diagnostics_df)


Train target distribution:


count    10000.000000
mean        76.942507
std         15.186669
min          0.000000
25%         66.937500
50%         77.810000
75%         88.472500
max        100.000000
Name: career_success_score, dtype: float64

Prediction diagnostics:


,name,mean,std,min,p01,p05,p25,p50,p75,p95,p99,max,n_equal_100,n_below_40,n_below_50
2,LightGBM_TFIDF,76.143308,12.606400,34.193814,46.972047,55.031901,67.123621,76.120326,85.448666,97.081571,100.000000,100.000000,220,16,186
4,CatBoost_TFIDF,76.213119,11.888095,39.430939,50.200477,56.860074,67.741359,75.758598,84.914998,96.415367,100.000000,100.000000,201,3,92
7,tree_light_blend,76.127524,11.840949,35.430466,48.547578,56.712427,67.860534,75.968263,84.562499,96.063735,100.000000,100.000000,108,8,134
6,balanced_blend,76.139497,11.815251,36.125832,48.803247,56.768190,67.888347,75.963513,84.589896,96.057276,100.000000,100.000000,108,7,127
1,ElasticNet_TFIDF,76.039290,11.789888,33.629288,48.086266,56.577530,68.072154,76.014950,84.072925,96.277684,100.000000,100.000000,254,14,153
0,Ridge_TFIDF,76.043469,11.779836,33.608551,48.096017,56.641022,68.085865,76.005721,84.064465,96.241760,100.000000,100.000000,251,14,150
5,conservative_ridge_blend,76.108256,11.764137,35.124605,48.848250,56.845603,67.942411,75.984566,84.432417,96.082831,100.000000,100.000000,108,8,137
3,XGBoost_TFIDF,76.229164,11.745897,35.850048,49.279819,57.150173,67.943268,75.893921,84.718575,96.287041,100.000000,100.000000,185,7,117
8,conservative_blend_year_smooth_5,76.120173,11.180966,37.076278,50.162264,57.814351,68.370130,76.004180,84.041338,95.116963,98.768226,98.897129,0,3,96
9,conservative_blend_year_smooth_10,76.132090,10.598270,39.027951,51.540746,58.749806,68.807902,76.032781,83.609028,94.152113,97.547243,97.794258,0,1,65


## 11. Submission notes

Suggested Kaggle submission order after CatBoost public score was worse than expected:

1. `submission_03_Ridge_TFIDF.csv`
2. `submission_03_conservative_ridge_blend.csv`
3. `submission_03_tree_light_blend.csv`
4. `submission_03_LightGBM_TFIDF.csv`
5. Optional: `submission_03_conservative_blend_year_smooth_5.csv`

Do not submit too many random variations.  
Use public score feedback carefully, but avoid overfitting to public leaderboard.


In [15]:
print("Submission files created:")
for file in sorted(os.listdir(SUBMISSION_DIR)):
    if file.startswith("submission_03_") and file.endswith(".csv"):
        print(os.path.join(SUBMISSION_DIR, file))


Submission files created:
../submissions\submission_03_CatBoost_TFIDF.csv
../submissions\submission_03_ElasticNet_TFIDF.csv
../submissions\submission_03_LightGBM_TFIDF.csv
../submissions\submission_03_Ridge_TFIDF.csv
../submissions\submission_03_XGBoost_TFIDF.csv
../submissions\submission_03_balanced_blend.csv
../submissions\submission_03_conservative_blend_year_smooth_10.csv
../submissions\submission_03_conservative_blend_year_smooth_15.csv
../submissions\submission_03_conservative_blend_year_smooth_5.csv
../submissions\submission_03_conservative_ridge_blend.csv
../submissions\submission_03_tree_light_blend.csv
